# Training Series: Introduction to eXtended Discontinuous Galerkin Methods for Multi-Phase Flow Problems <br> - Hands-On Worksheets

In [ ]:
#r "../BoSSSpad/BoSSSpad.dll"
//#r "C:\Program Files (x86)\FDY\BoSSS\bin\Release\net5.0\BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Platform.LinAlg;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.Grid.RefElements;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

## Hands-On 3: Poisson Equation and the Symmetric Interior Penalty Flux 

## Symmteric Interior Penalty

### Problem statement

We consider the 2D Poisson problem:
$$ \Delta u = f(x,y) $$

where $f(x,y) \neq 0$ is an arbitrary function of $x$ and/or $y$.
Within this exercise, we are going to investigate the Symmetric Interior Penalty discretization method (SIP) for the Laplace operator

$$a_{\text{sip}}(u,v)
= \int_{\Omega} \underbrace{\nabla u \cdot \nabla v}_{\text{Volume\ term}}dV
  - \oint_{\Gamma \setminus \Gamma_{N }} \underbrace{
        \{\nabla u\} \cdot n_{\Gamma} \llbracket v \rrbracket
     }_{\text{consistency term}} + \underbrace{
        \{\nabla v\} \cdot \vec{n}_{\Gamma} \llbracket u \rrbracket
     }_{\text{symmetry term}} dA
  + \oint_{\Gamma \setminus \Gamma_{N}} \underbrace{
       \eta \llbracket u \rrbracket \llbracket v \rrbracket
    }_{\text{penalty term}} dA$$

The use of these fluxes including a penalty term stabilizes the DG-discretization for the Laplace operator.

### Implementation of the SIP fluxes

We need the following packages:

In [ ]:
using ilPSP.LinSolvers;
using ilPSP.Connectors.Matlab;
using NUnit.Framework;

First, we need a class in which the integrands are defined.
This also includes some technical aspects like the *TermActivationFlags*.

In [ ]:
public class SipLaplace :
        BoSSS.Foundation.IEdgeForm,   // edge integrals
        BoSSS.Foundation.IVolumeForm, // volume integrals
        IEquationComponentCoefficient // update of coefficients (e.g. length scales) required for penalty parameters 
{
    /// We do not use parameters (e.g. variable viscosity, ...)
    /// at this point: so this can be null
    public IList<string> ParameterOrdering { 
        get { return new string[0]; } 
    } 
    /// but we have one argument variable, $u$ (our trial function)
    public IList<String> ArgumentOrdering { 
        get { return new string[] { "u" }; } 
    }
    /// The \code{TermActivationFlags} tell \BoSSS which kind of terms, 
    /// i.e. products of u, v, \nabla u, and \nabla v
    /// the VolumeForm(...) actually contains.
    /// This additional information helps to improve the performance.
    public TermActivationFlags VolTerms {
        get {
            return TermActivationFlags.GradUxGradV;
        }
    }
    /// activation flags for the 'InnerEdgeForm(...)'
    public TermActivationFlags InnerEdgeTerms {
        get {
            return (TermActivationFlags.AllOn);
            // if we do not care about performance, we can activate all terms.
        }
    }
    public TermActivationFlags BoundaryEdgeTerms {
       get {
           return TermActivationFlags.AllOn;
        }
    }

    /// For the computation of the penalty factor $\eta$,
    /// we require some length scale for each cell and 
    /// the polynomial degree of the DG approximation.
    MultidimensionalArray cj;

    double penalty_base; // base factor must be scaled by polynomial degree  

    /// The additional scaling of the penalty by polynomial degree 
    /// and in dependence of geometry can be obtained through 
    /// implementing the \code{IEquationComponentCoefficient} interface:
    public void CoefficientUpdate(CoefficientSet cs, int[] DomainDGdeg, int TestDGdeg) {
        int D = cs.GrdDat.SpatialDimension;
        double _D = D;
        double _p = DomainDGdeg.Max();

        double penalty_deg_tri = (_p + 1) * (_p + _D) / _D; // formula for triangles/tetras
        double penalty_deg_sqr = (_p + 1.0) * (_p + 1.0); // formula for squares/cubes

        penalty_base = Math.Max(penalty_deg_tri, penalty_deg_sqr); // the conservative choice
        //Console.WriteLine("Setting penalty base factor for deg " + _p + " to " + penalty_base);

        cj = ((GridData)(cs.GrdDat)).Cells.cj; // inverse length scale (dimension is one-over-length, resp. area over volume)
    }     
    
            
    /// The safety factor for the penalty factor should be in the order of 1.
    /// A very large penalty factor increases the condition number of the system, but without affecting stability.
    /// A very small penalty factor yields to an unstable discretization.
    public double PenaltySafety = 2.2; 

    /// The actual computation of the penalty factor, which should be used in the \code{InnerEdgeForm} and \code{BoundaryEdgeForm} functions.
    /// Hint: for the parameters \code{jCellIn}, \code{jCellOut} and \code{g}, take a look at \code{CommonParams} and \code{CommonParamsBnd}.
    double PenaltyFactor(int jCellIn, int jCellOut) {
        double cj_in         = cj[jCellIn];
        double eta           = penalty_base * cj_in * PenaltySafety;
        if(jCellOut >= 0) {
            double cj_out = cj[jCellOut];
            eta           = Math.Max(eta, penalty_base * cj_out * PenaltySafety);
        }
        
        return eta;
    }
    
    /// The following functions cover the actual math.
    /// For any discretization of the Laplace operator, we have to specify:
    /// 
    /// - a volume integrand,
    /// - an edge integrand for inner edges, i.e. on $\Gamma_i$,
    /// - an edge integrand for boundary edges, i.e. on $\partial \Omega$.
    /// 
    /// The integrand for the volume integral:
    public double VolumeForm(ref CommonParamsVol cpv, 
           double[] U, double[,] GradU, // the trial-function u (i.e. the function we search for) and its gradient
           double V, double[] GradV     // the test function; 
           ) {
 
        // == TODO == add the SIP volume form 
        double acc = 0.0;
        for(int d = 0; d < cpv.D; d++)
            acc += GradU[0, d] * GradV[d];
        return acc;
    }

    /// The integrand for the integral on the inner edges,
    /// 
    ///   -( {\nabla u} [[v]]) \cdot \vec{n}_{\Gamma} 
    ///   -( {\nabla v} [[u]]) \cdot \vec{n}_{\Gamma} 
    ///   + \eta [[u]] [[v]] :
    /// 
    public double InnerEdgeForm(ref CommonParams inp, 
        double[] U_IN, double[] U_OT, double[,] GradU_IN, double[,] GradU_OT, 
        double V_IN, double V_OT, double[] GradV_IN, double[] GradV_OT) {
 
        double eta = PenaltyFactor(inp.jCellIn, inp.jCellOut);
 
        double Acc = 0.0;
        // == TODO == add the consistency term : -({ \/u } [[ v ]])*Normal
        // == TODO == add the symmetry term: : -({ \/v } [[ u ]])*Normal
        // == TODO == add the penalty term: eta*[[u]]*[[v]]

        for(int d = 0; d < inp.D; d++) { // loop over vector components 
            // consistency term: -({ \/u } [[ v ]])*Normal
            // index d: spatial direction
            Acc -= 0.5 * (GradU_IN[0, d] + GradU_OT[0, d])*(V_IN - V_OT)
                       * inp.Normal[d];
 
            // symmetry term: -({ \/v } [[ u ]])*Normal
            Acc -= 0.5 * (GradV_IN[d] + GradV_OT[d])*(U_IN[0] - U_OT[0])
                       * inp.Normal[d];
        }
 
        // penalty term: eta*[[u]]*[[v]]
        Acc += eta*(U_IN[0] - U_OT[0])*(V_IN - V_OT);
        
        return Acc;
    } 

    /// The integrand on boundary edges, i.e. on $\partial \Omega$, is
    ///  
    ///   -( {\nabla u} [[v]]) \cdot \vec{n}_{\Gamma} 
    ///   -( {\nabla v} [[u]]) \cdot \vec{n}_{\Gamma} 
    ///   +  \eta [[u]]  [[v]] .
    /// 
    /// For the boundary we have to consider the special definition for 
    /// the mean-value operator ${-}$ and the jump operator $[[-]]$ on the boundary.
    public double BoundaryEdgeForm(ref CommonParamsBnd inp, 
        double[] U_IN, double[,] GradU_IN, double V_IN, double[] GradV_IN) {
 
        double eta = PenaltyFactor(inp.jCellIn, -1);
        
        double Acc = 0.0;
        // == TODO == add the consistency term: -({ \/u } [[ v ]])*Normale
        // == TODO == add the symmetry term: -({ \/v } [[ u ]])*Normale
        // == TODO == add the penalty term: eta*[[u]]*[[v]]

        for(int d = 0; d < inp.D; d++) { // loop over vector components 
            // consistency term: -({ \/u } [[ v ]])*Normale
            // index d: spatial direction
            Acc -= (GradU_IN[0, d])*(V_IN) * inp.Normal[d];
 
            // symmetry term: -({ \/v } [[ u ]])*Normale
            Acc -= (GradV_IN[d])*(U_IN[0]) * inp.Normal[d];
        }
 
        // penalty term: eta*[[u]]*[[v]]
        Acc += eta*(U_IN[0])*(V_IN);
 
        return Acc;
    }
}

### Evaluation of the Poisson operator in 1D

We consider the following problem:
$$
\Delta u = 2,\quad -1<x<1,\quad u(-1)=u(1)=0.
$$
The solution is $u_{ex}(x) = 1 - x^2$. Since this is quadratic, we can represent it **exactly** in a DG space of order 2.
As usual, we have to set up a grid, a basis and a right-hand-side.

In [ ]:
var grd1D                     = Grid1D.LineGrid(GenericBlas.Linspace(-1,1,10));
var DGBasisOn1D               = new Basis(grd1D, 2);
var RHS                       = new SinglePhaseField(DGBasisOn1D, "RHS");
RHS.ProjectField((double x) => 2);

In [ ]:
var i_SipLaplace              = new SipLaplace();
var Operator_SipLaplace       = i_SipLaplace.Operator();

We now want to calculate the residual after inserting the exact solution as well as a wrong solution. 
The implementation of the exact solution:

In [ ]:
var u_ex         = new SinglePhaseField(DGBasisOn1D, "$u_{ex}$");
u_ex.ProjectField((double x) => 1.0 - x*x);

The implementation of a spurious, i.e. a wrong solution; we take the exact solution and add random values in each cell:

In [ ]:
var u_wrong      = new SinglePhaseField(DGBasisOn1D, "$u_{wrong}$");
u_wrong.ProjectField((double x) => 1.0 - x*x);
Random R         = new Random();
for(int j = 0; j < grd1D.GridData.Cells.NoOfLocalUpdatedCells; j++){
    double ujMean = u_wrong.GetMeanValue(j);
    ujMean += R.NextDouble();
    u_wrong.SetMeanValue(j, ujMean);
}

Evaluating the Laplace operator using the different solutions:

In [ ]:
var Residual     = new SinglePhaseField(DGBasisOn1D,"Resi1");
var ResidualNorm = new List<double>();
foreach(var u in new DGField[] {u_ex, u_wrong}) {
    Residual.Clear();
    Operator_SipLaplace.Evaluate(u, Residual);  // evaluate
    Residual.Acc(-1.0, RHS);    
    double ResiNorm = Residual.L2Norm();
    ResidualNorm.Add(ResiNorm);
    Console.WriteLine("Residual for " + u.Identification + " = " + ResiNorm);  
}

Residual for $u_{ex}$ = 1.3694083221572641E-12
Residual for $u_{wrong}$ = 1077.2166538528452


In [ ]:
Assert.LessOrEqual(ResidualNorm[0], 1e-10);
Assert.GreaterOrEqual(ResidualNorm[1], 1e-1);

### The penalty parameter of the SIP and stability in 2D

We define a two-dimensional grid:

In [ ]:
var grd2D       = Grid2D.Cartesian2DGrid(GenericBlas.Linspace(-1,1,21), 
                                         GenericBlas.Linspace(-1,1,16));
var DGBasisOn2D = new Basis(grd2D, 5);
var Mapping2D   = new UnsetteledCoordinateMapping(DGBasisOn2D);

We consider the example 
$$
    -\Delta u = \pi^2 (a_x^2 + a_y^2)/4 \cos(a_x \pi x/2) \cos(a_y \pi y/2) 
      \text{ with } 
      (x,y) \in (-1,1)^2
$$
and $u = 0$ on the boundary.
The exact solution is $u_{Ex}(x,y) = \cos(a_x \pi  x/2) \cos(a_x \pi y/2)$, where $a_x$ and $a_y$ must be odd numbers
to comply with homogeneous bounary condition.

In [ ]:
double ax = 1.0; // must be an odd number to comply with homogeneous Dirichlet boundary condition
double ay = 3.0; // must be an odd number to comply with homogeneous Dirichlet boundary condition
Func<double[], double> exSol = 
        (X => Math.Cos(X[0]*ax*Math.PI*0.5)*Math.Cos(X[1]*ay*Math.PI*0.5));
Func<double[], double> exRhs = 
        (X => ((ax.Pow2() + ay.Pow2())/4.0)*Math.PI.Pow2()
             *Math.Cos( X[0]*ax*Math.PI*0.5 )*Math.Cos( X[1]*ay*Math.PI*0.5 )); // == - /\ exSol

SinglePhaseField RHS = new SinglePhaseField(DGBasisOn2D, "RHS");
RHS.ProjectField(exRhs);
double[] RHSvec = RHS.CoordinateVector.ToArray();

We check our discretization once more in 2D; the residual should be low,
but not exactly (resp. up to $10^{-12}$) since the solution is not 
polynomial and cannot be fulfilled exactly.

In [ ]:
SinglePhaseField u = new SinglePhaseField(DGBasisOn2D,"u");
u.ProjectField(exSol);

var Matrix_SIP_sf = Operator_SipLaplace.ComputeMatrix(Mapping2D, null, Mapping2D);

SinglePhaseField Residual = new SinglePhaseField(DGBasisOn2D,"Residual");
Residual.Acc(1.0, RHS);
Matrix_SIP_sf.SpMV(-1.0, u.CoordinateVector, 1.0, Residual.CoordinateVector);
Console.WriteLine("Residual L2 norm: " + Residual.L2Norm());

Residual L2 norm: 0.02104929706815022


In [ ]:
//== Task == Above, how could you decrease the Residual L2 norm?

We also check that the matrix is symmetric:

In [ ]:
var checkMatrix = Matrix_SIP_sf - Matrix_SIP_sf.Transpose();
checkMatrix.InfNorm()

5.837641481321043E-10

In [ ]:
Assert.LessOrEqual(checkMatrix.InfNorm(), 1e-8);

### Matrix properties for different penalty factors (Bonus)

We are going to choose the **PenaltySafety** for the **SipLaplace**
from the following list

In [ ]:
double[] SFs = new double[] {0.001, 0.002, 0.01, 0.02, 0.1, 0.2, 1, 2, 10, 20, 100};

Now, we assemble the matrix of the SIP for different 
**PenaltySafety**-factors. We also try to solve the linear system
using an iterative method.

 As Matlab is called multiple times during this 
command, it can take some minutes until it is done.

In [ ]:
int cnt     = 0;
var Results = new List<(double safetyFactor, double condNumber, int NoOfIterations, double L2errror, bool isDefinite)>();
foreach(double sf in SFs) {

    cnt++;
    i_SipLaplace.PenaltySafety    = sf;
    var Matrix_SIP_sf             = Operator_SipLaplace.ComputeMatrix(
                                    Mapping2D, null, Mapping2D);
    double condNo1                = Matrix_SIP_sf.condest();  
    bool definite                 = Matrix_SIP_sf.IsDefinite();
 
    /// We solve the system 
    /// 
    ///     Matrix\_SIP\_sf \cdot u =  RHS
    /// 
    /// using a an iterative solver, the so-called conjugate gradient (CG) method.
    /// CG requires a positive definite matrix. 
    /// The function \code{Solve\_CG} returns the number of iterations.
    SinglePhaseField u = new SinglePhaseField(DGBasisOn2D,"u");
    u.InitRandom();
    int NoOfIter = Matrix_SIP_sf.Solve_CG(u.CoordinateVector, RHSvec);
 
    SinglePhaseField Error = new SinglePhaseField(DGBasisOn2D,"Error");
    Error.ProjectField(exSol);
    Error.Acc(-1.0, u);
 
    double L2err = u.L2Error(exSol);
 
    Console.WriteLine(sf + "\t" + condNo1.ToString("0.#E-00") 
                         + "\t" + NoOfIter 
                         + "\t" + L2err.ToString("0.#E-00") 
                         + "\t" + definite);
    Results.Add((sf, condNo1, NoOfIter, L2err, definite));
}

0.001	8.1E04	13306	4E-07	False
0.002	8.1E04	14252	5.4E-07	False
0.01	7.9E04	21004	2.4E-07	False
0.02	7.9E04	25469	2.6E-07	False
0.1	8.5E05	31584	3.8E-06	False
0.2	4.5E04	4815	2.3E-07	False
1	3.5E05	1428	2.4E-06	True
2	7.9E05	2085	3.6E-06	True
10	4.3E06	4510	1.3E-05	True
20	8.6E06	6141	3.1E-05	True
100	4.3E07	12283	1.5E-04	True


In [ ]:
foreach(var r in Results) {
    if(r.safetyFactor >= 1 && r.safetyFactor <= 20) {
        Assert.LessOrEqual(r.condNumber, 1e7); // cond No.
        Assert.LessOrEqual(r.NoOfIterations, 7000); // iter
        Assert.LessOrEqual(r.L2errror, 1e-4); // L2 err
        Assert.IsTrue(r.isDefinite); // definite   
    }
    if(r.Item1 <= 0.1) {
        Assert.IsFalse(r.isDefinite); // indefinite   
    }
}

Plot the number of conjugate gradient iterations versus the **PenaltySafety**.

In [ ]:
var xValues = Results.Select(r => r.safetyFactor).ToArray();
var yValues = Results.Select(r => ((double)(r.NoOfIterations))).ToArray();

var plt = new Plot2Ddata();
plt.AddDataGroup(xValues, yValues);

/// A logarithmic scale is used for the horizontal axis.
plt.LogX = true;

/// Set Format
plt.dataGroups[0].Format =  new PlotFormat(lineColor: LineColors.Blue, 
                                           pointSize: 2, 
                                           dashType: DashTypes.DotDashed, 
                                           Style: Styles.LinesPoints, 
                                           pointType:PointTypes.OpenCircle);
// Show!
plt.PlotNow()

Using gnuplot: C:\Users\smuda\AppData\Local\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 5000 
 
 
 
 
 10000 
 
 
 
 
 15000 
 
 
 
 
 20000 
 
 
 
 
 25000 
 
 
 
 
 30000 
 
 
 
 
 35000 
 
 
 
 
 $10 -3 $ 
 
 
 
 
 $10 -2 $ 
 
 
 
 
 $10 -1 $ 
 
 
 
 
 $10 0 $ 
 
 
 
 
 $10 1 $ 
 
 
 
 
 $10 2 $ 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 gnuplot_plot_1

### Convergence study, indefinite vs. definite.

We are going to solve the SIP-system for different grid resolutions,
comparing an insufficient penalty (*PenaltySafety*-factor $\eta_0 = 0.01$) to a penalty which is large enough ($\eta_0 = 2$).

In [ ]:
double[] Resolution = new double[] { 2, 4, 8, 16, 32, 64 };
List<double> L2Error_indef  = new List<double>();
List<double> L2Error_posdef = new List<double>();
int cnt = 0;
foreach(int Res in Resolution) {
    cnt++;
    //var grd2D = Grid2D.UnstructuredTriangleGrid(GenericBlas.Linspace(-1,1,(int)Res + 1), 
    //                                            GenericBlas.Linspace(-1,1,(int)Res + 1));
    var grd2D = Grid2D.Cartesian2DGrid(GenericBlas.Linspace(-1,1,(int)Res + 1), 
                                      GenericBlas.Linspace(-1,1,(int)Res + 1));
    var gdata2D = new GridData(grd2D);
    var DGBasisOn2D = new Basis(gdata2D, 2);
    var Mapping2D  = new UnsetteledCoordinateMapping(DGBasisOn2D);
 
    SinglePhaseField RHS = new SinglePhaseField(DGBasisOn2D, "RHS");
    RHS.ProjectField(exRhs);
    SinglePhaseField uEx = new SinglePhaseField(
           new Basis(gdata2D, DGBasisOn2D.Degree*2),
           "Error");
    uEx.ProjectField(exSol);
 
 
    i_SipLaplace.PenaltySafety    = 0.01; // == TODO == choose a insufficient small penalty parameter (indefinite matrix)
    var Matrix_SIP_indef          = Operator_SipLaplace.ComputeMatrix(
                                    Mapping2D,null,Mapping2D);
 
    SinglePhaseField u_indef = new SinglePhaseField(DGBasisOn2D,"u_indef");
    Matrix_SIP_indef.Solve_Direct(u_indef.CoordinateVector, 
                                  RHS.CoordinateVector);
    var Error_indef = uEx.CloneAs();
    Error_indef.AccLaidBack(-1.0, u_indef);
    L2Error_indef.Add(Error_indef.L2Norm());
 

    i_SipLaplace.PenaltySafety    = 2.0; // == TODO == choose a sufficient large penalty parameter (definite matrix)
    var Matrix_SIP_posdef         = Operator_SipLaplace.ComputeMatrix(
                                    Mapping2D, null, Mapping2D);
 
    SinglePhaseField u_posdef = new SinglePhaseField(DGBasisOn2D,"u_posdef");
    Matrix_SIP_posdef.Solve_Direct(u_posdef.CoordinateVector, 
                                   RHS.CoordinateVector);
    var Error_posdef = uEx.CloneAs();
    Error_posdef.AccLaidBack(-1.0, u_posdef);
    L2Error_posdef.Add(Error_posdef.L2Norm());
    
    //Tecplot("ConvStudy-" + cnt, uEx, u_posdef, u_indef); // activate this line for plotting!
    
    Console.WriteLine(L2Error_indef.Last().ToString("0.#E-00") 
                      + "\t" + L2Error_posdef.Last().ToString("0.#E-00"));
}

1.5E01	7.2E-01
4.1E-01	1.9E-01
3.1E-02	1.7E-02
5.8E-03	1.6E-03
8.4E-04	1.7E-04
1.1E-04	2.1E-05


### Convergence Plot and Conclusions

The convergence plot should unveil that there is something wrong if the
penalty factor is set too low.
Unfortunately, **it does not**, so this is some kind of **anti-example**;
It is in this tutorial anyway **to illustrate the difficulties of numerical testing**.

The reason why the indefinite matrix still gives a solution convergence 
is very likely that the solver which is used in BoSSS is also (sometimes) capable
of solving singular or close-to-singular systems, i.e. systems without a unique solution.
In those cases, it selects a solution with a minimal solution norm. 
Since BoSSS uses an orthonormal basis the $L^2$ norm of the DG-Field is identical to the $l_2$ norm of the 
coordinate vector (Parseval's identity).
Therefore, the solver by chance adds additional stability which is not part of the (instable) discretization.

The error of the positive definite system, *Error\_posdef*, where the 
penalty is chosen sufficiently large converges with the expected 
rate.

In [ ]:
var plt = new Plot2Ddata();
plt.AddDataGroup("indef mtx", Resolution, L2Error_indef);
plt.AddDataGroup("pos def mtx", Resolution, L2Error_posdef);

/// A double-logarithmic scale is used:
plt.LogX = true;
plt.LogY = true;

/// Set Format
plt.dataGroups[0].Format =  new PlotFormat(lineColor: LineColors.Red, 
                                           pointSize: 2, 
                                           dashType: DashTypes.DotDashed, 
                                           Style: Styles.LinesPoints, 
                                           pointType:PointTypes.OpenCircle);
plt.dataGroups[1].Format =  new PlotFormat("Blue-.o"); // altenatively, using MATLAB-like format strings
plt.dataGroups[1].Format.PointSize = 2;
// Show!
plt.PlotNow()

Using gnuplot: C:\Users\smuda\AppData\Local\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 $10 -5 $ 
 
 
 
 
 $10 -4 $ 
 
 
 
 
 $10 -3 $ 
 
 
 
 
 $10 -2 $ 
 
 
 
 
 $10 -1 $ 
 
 
 
 
 $10 0 $ 
 
 
 
 
 $10 1 $ 
 
 
 
 
 $10 2 $ 
 
 
 
 
 $10 0 $ 
 
 
 
 
 $10 1 $ 
 
 
 
 
 $10 2 $ 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 indef mtx 
 
 
 indef mtx 
 
 
 
 
 
 
 
 
 
 
 
 
 pos def mtx 
 
 
 pos def mtx

Finally, we are going to compute the convergence rate of the SIP
discretization.

We compute the slope of the log-log plot:

In [ ]:
double dk = Resolution.LogLogRegressionSlope(L2Error_posdef);
dk

-3.1159715513644644

In [ ]:
Assert.LessOrEqual(dk, -2.9);

## Poisson Equation as a System (Bonus)

### Problem statement

Within this exercise, we are going to investigate 
the discretization of a Poisson equation as a system.
Obviously, it is possible to discretize the Poisson equation as a system of
first-order-PDE's, introducing a vector field $\vec{\sigma}$:
$$
\begin{align}
 \vec{\sigma}  + \nabla u & = 0, & & \text{ in } \Omega
 \\
 \operatorname{div}(\vec{\sigma}) &  = g_{\Omega}, & & \text{ in } \Omega
 \\
  u                                               & = g_D, & & \text{ on } \Gamma_D \\
  - \vec{\sigma} \cdot \vec{n}_{\partial \Omega} & = g_N, & & \text{ on } \Gamma_N
\end{align}
$$
resp. in matrix-notation:
$$
\begin{align*}
  \begin{bmatrix}
    1 & \nabla \\
    \operatorname{div} & 0 \\
  \end{bmatrix}\cdot
  \begin{bmatrix}
    \vec{\sigma}\\
    u
  \end{bmatrix}=
  \begin{bmatrix}
    0 \\
    g_{\Omega}
  \end{bmatrix}
\end{align*}
$$
This exercise, together with the previous one, will form the foundation for an incompressible Stokes- resp. Navier-Stokes solver. 

### Tests on the divergence

We are going to implement two different formulations of the 
divergence-operator for which going to show equivalence. 
We implement a common base-class for both formulations:

In [ ]:
abstract public class BaseDivergence :  
        BoSSS.Foundation.IEdgeForm, // edge integrals 
        BoSSS.Foundation.IVolumeForm     // volume integrals 
{ 
    /// We don't use parameters (e.g. variable viscosity, ...)
    /// at this point: so the parameter list can be null, resp. empty:
    public IList<string> ParameterOrdering {  
        get { return null; }  
    } 
 
    /// But we have a vector argument variable, 
    /// $ [ \sigma_1, \sigma_2 ] = \vec{\sigma} $
    /// (our trial function):
    public IList<String> ArgumentOrdering {  
        get { return new string[] { "sigma1", "sigma2" }; }  
    } 
 
    public TermActivationFlags VolTerms { 
        get { 
            return TermActivationFlags.AllOn; 
        } 
    } 
 
    public TermActivationFlags InnerEdgeTerms { 
        get { 
            return (TermActivationFlags.AllOn);  
        } 
    } 
 
    public TermActivationFlags BoundaryEdgeTerms { 
       get { 
           return TermActivationFlags.AllOn; 
        } 
    } 
 
    /// The following functions cover the actual math.
    /// For any discretization of the divergence-operator, we have to specify:
    /// \begin{itemize}
    ///    \item a volume integrand,
    ///    \item an edge integrand for inner edges, i.e. on $ \Gamma_i$,
    ///    \item an edge integrand for boundary edges, 
    ///          i.e. on $\partial \Omega$.
    /// \end{itemize}
    /// These functions are declared as \code{abstract}, meaning that one has 
    /// to specify them in classes derived from \code{BaseLaplace}.
 
    abstract public double VolumeForm(ref CommonParamsVol cpv,  
           double[] U, double[,] GradU,  
           double V, double[] GradV);         
 
    abstract public double InnerEdgeForm(ref CommonParams inp,  
        double[] U_IN, double[] U_OT, double[,] GradU_IN, double[,] GradU_OT,  
        double V_IN, double V_OT, double[] GradV_IN, double[] GradV_OT); 
 
    abstract public double BoundaryEdgeForm(ref CommonParamsBnd inp,  
        double[] U_IN, double[,] GradU_IN, double V_IN, double[] GradV_OT); 
}

In [ ]:
/// We are going to use both, Dirichlet- and Neumann-boundary conditions in this exercise; 
/// the function \code{IsDirichletBndy} is used to specify the type of boundary condition at point \code{X}:
static class BndyMap {
static public Func<double[],bool> IsDirichletBndy = delegate(double[] X) { 
    double x = X[0]; 
    double y = X[1]; 
    if(Math.Abs(x - (-1.0)) < 1.0e-8) 
        return true;     
    if(Math.Abs(y - (-1.0)) < 1.0e-8) 
        return true;     
    return false; 
};
}

**Formulation (i): Central-difference-form of $\text{div}$**

Reminder: central-difference form
>$$ \hat{\vec{\sigma}} \cdot \vec{n} = \begin{cases} \{\vec{\sigma}\} \cdot \vec{n} \quad \textrm{ on } \Gamma_i \\ 
\vec{\sigma}^- \cdot \vec{n} \quad \textrm{ on } \Gamma_{\textrm{D}} \\ 
g_{\textrm{N}} \quad \textrm{ on } \Gamma_{\textrm{N}} \\ \end{cases}$$

The implementation of the form is as follows:

In [ ]:
class Divergence_cendiff : BaseDivergence { 
 
 
    /// The volume form is equal to 
    /// $ -\vec{\sigma} \cdot \nabla v$:
    override public double VolumeForm(ref CommonParamsVol cpv,  
        double[] Sigma, double[,] GradSigma,  
        double V, double[] GradV) { 
        
        double Acc = 0; 
        // == TODO == add the volume term
        for(int d = 0; d < cpv.D; d++) { 
            Acc -= Sigma[d]*GradV[d]; 
        } 
        return Acc; 
    } 
 
    /// At the cell boundaries, we use a central-difference-flux,
    /// i.e. $\mean{\vec{\sigma}} \cdot \vec{n}_{\Gamma} \jump{v}$:
    override public double InnerEdgeForm(ref CommonParams inp,  
        double[] Sigma_IN, double[] Sigma_OT, double[,] GradSigma_IN, double[,] GradSigma_OT,  
        double V_IN, double V_OT, double[] GradV_IN, double[] GradV_OT) { 
 
        double Acc = 0; 
        // == TODO == add the cental-difference inner edge form
        for(int d = 0; d < inp.D; d++) { 
            Acc += 0.5*(Sigma_IN[d] + Sigma_OT[d])*inp.Normal[d]*(V_IN - V_OT); 
        } 
        return Acc; 
    } 
 
    override public double BoundaryEdgeForm(ref CommonParamsBnd inp,  
        double[] Sigma_IN, double[,] GradSigma_IN, double V_IN, double[] GradV_OT) { 
 
        double Acc = 0; 
        // == TODO == add the cental-difference boundary edge forms (gNeu = 0.0)
        if(BndyMap.IsDirichletBndy(inp.X)) { 
            /// Dirichlet-boundary: by taking the inner value of $\vec{\sigma}$, 
            /// this is a free boundary with respect to $\vec{\sigma}$.
            for(int d = 0; d < inp.D; d++) { 
                Acc += Sigma_IN[d]*inp.Normal[d]*V_IN; 
            } 
        } else { 
            /// Neumann-boundary
            double gNeu = 0.0; 
            Acc += gNeu*V_IN; 
        } 
        return Acc; 
    } 
}

**Formulation (ii): 'Strong' form of $\text{div}$**

Here, we use the form 
$$
   b(\vec{\sigma},v) = 
   \oint_{\Gamma \backslash \Gamma_D} 
          \{v\} \llbracket \vec{\sigma} \rrbracket \cdot \vec{n}_\Gamma dA 
   - \int_{\Omega} \text{div}(\vec{\sigma}) \cdot v dV
$$
This is actually the negative divergence, which will be more useful later on.

In [ ]:
class Divergence_strong : BaseDivergence { 
 
    /// We have to implement \code{VolumeForm},
    /// \emph{InnerEdgeForm} and \code{BoundaryEdgeForm}:
    override public double VolumeForm(ref CommonParamsVol cpv,  
        double[] Sigma, double[,] GradSigma,  
        double V, double[] GradV) { 
            
        double Acc = 0; 
        // == TODO == add the volume term
        for(int d = 0; d < cpv.D; d++) { 
            Acc -= GradSigma[d,d]*V; 
        } 
        return Acc; 
    } 
 
    override public double InnerEdgeForm(ref CommonParams inp,  
        double[] Sigma_IN, double[] Sigma_OT, double[,] GradSigma_IN, double[,] GradSigma_OT,  
        double V_IN, double V_OT, double[] GradV_IN, double[] GradV_OT) { 
 
        double Acc = 0; 
        // == TODO == add the inner edge term of the strong div form
        for(int d = 0; d < inp.D; d++) { 
            Acc += 0.5*(V_IN + V_OT)*(Sigma_IN[d] - Sigma_OT[d])*inp.Normal[d]; 
        } 
        return Acc; 
    } 
 
    override public double BoundaryEdgeForm(ref CommonParamsBnd inp,  
        double[] Sigma_IN, double[,] GradSigma_IN, double V_IN, double[] GradV_OT) { 
 
        double Acc = 0; 
        // == TODO == add the boundary edge term of the strong div form (gNeu = 0.0)
        if(BndyMap.IsDirichletBndy(inp.X)) { 
            Acc = 0;
        } else { 
            double gNeu = 0.0; 
            for(int d = 0; d < inp.D; d++) { 
                Acc += Sigma_IN[d]*inp.Normal[d]*V_IN; 
            } 
            Acc -= gNeu*V_IN; 
        } 
        return Acc; 
    } 
}

We are going to test the equivalence of both formulations on a 2D grid, using a DG basis of degree 1:

In [ ]:
var grd2D               = Grid2D.Cartesian2DGrid(GenericBlas.Linspace(-1,1,6),                                                    
                                                 GenericBlas.Linspace(-1,1,7)); 
var b                   = new Basis(grd2D, 1); 
SinglePhaseField sigma1 = new SinglePhaseField(b,"sigma1"); 
SinglePhaseField sigma2 = new SinglePhaseField(b,"sigma2"); 
CoordinateVector sigma  = new CoordinateVector(sigma1,sigma2); 
var TrialMapping        = sigma.Mapping; 
var TestMapping         = new UnsetteledCoordinateMapping(b);

In [ ]:
/// We create the matrix of the central-difference formulation:
var OpDiv_cendiff = (new Divergence_cendiff()).Operator(); 
var MtxDiv_cendiff = OpDiv_cendiff.ComputeMatrix(TrialMapping,  
                                                 null,  
                                                 TestMapping);

We create the matrix of the strong formulation and show that the matrices of both formulations are equal.

We use the \code{InfNorm(...)}-method to identify whether a matrix is (approximately) zero or not.

In [ ]:
var OpDiv_strong  = (new Divergence_strong()).Operator(); 
var MtxDiv_strong = OpDiv_strong.ComputeMatrix(TrialMapping, null, TestMapping); 
var TestP         = MtxDiv_cendiff + MtxDiv_strong; 
TestP.InfNorm()

3.552713678800501E-15

### The gradient-operator

For the variational formulation of the gradient operator, a vector-valued
test-function is required. Unfourtunately, this is not supported by 
*BoSSS*. Therefore we have to discretize the gradient component-wise,
i.e. as $\partial_{x}$ and $\partial_y$. 

A single derivative can obviously be expressed as a divergence by the
identity $ \partial_{x_d} = \text{div}( \vec{e}_d u ) $.

In [ ]:
class Gradient_d : 
        BoSSS.Foundation.IEdgeForm, // edge integrals 
        BoSSS.Foundation.IVolumeForm     // volume integrals 
{ 
    public Gradient_d(int _d) { 
        this.d = _d; 
    } 
 
    /// The component index of the gradient:
    int d; 
 
    /// As ususal, we do not use parameters:
    public IList<string> ParameterOrdering {  
        get { return null; }  
    } 
 
    /// We have one argument $u$:
    public IList<String> ArgumentOrdering {  
        get { return new string[] { "u" }; }  
    } 
 
    public TermActivationFlags VolTerms { 
        get { return TermActivationFlags.AllOn; } 
    } 
 
    public TermActivationFlags InnerEdgeTerms { 
        get { return (TermActivationFlags.AllOn); } 
    } 
 
    public TermActivationFlags BoundaryEdgeTerms { 
       get { return TermActivationFlags.AllOn; } 
    } 
 
    /// Now, we implement 
    /// \begin{itemize}
    ///    \item the volume form $u \vec{e}_d \cdot \nabla v$
    ///    \item the boundary form 
    ///       ${u \ \vec{e}_d} \cdot \vec{n}_\Gamma [[v]]$
    /// \end{itemize}
    public double VolumeForm(ref CommonParamsVol cpv,  
           double[] U, double[,] GradU,  
           double V, double[] GradV) { 
 
        double Acc = 0; 
        // == TODO == add the volume form
        Acc -= U[0]*GradV[this.d]; 
        return Acc; 
    }         
 
    public double InnerEdgeForm(ref CommonParams inp,  
        double[] U_IN, double[] U_OT, double[,] GradU_IN, double[,] GradU_OT,  
        double V_IN, double V_OT, double[] GradV_IN, double[] GradV_OT) { 
 
        double Acc = 0; 
        // == TODO == add the inner edge form
        Acc += 0.5*(U_IN[0] + U_OT[0])*inp.Normal[this.d]*(V_IN - V_OT); 
        return Acc;   
     } 
 
    public double BoundaryEdgeForm(ref CommonParamsBnd inp,  
        double[] U_IN, double[,] GradU_IN, double V_IN, double[] GradV_OT) { 
 
        double Acc = 0; 
        // == TODO == add the boundary edge form (u_Diri = 0.0)
        if(BndyMap.IsDirichletBndy(inp.X)) { 
            double u_Diri = 0.0; 
            Acc += u_Diri*inp.Normal[this.d]*V_IN; 
        } else { 
            Acc += U_IN[0]*inp.Normal[this.d]*V_IN; 
        } 
        return Acc;               
    } 
}

Now, we are ready to assemble the full $\nabla$ operator
as $\left[ \begin{array}{c} \partial_x \\ \partial_y \end{array} \right]$.

In [ ]:
var OpGrad = new SpatialOperator(1,2,QuadOrderFunc.Linear(),"u","c1","c2"); 
OpGrad.EquationComponents["c1"].Add(new Gradient_d(0)); 
OpGrad.EquationComponents["c2"].Add(new Gradient_d(1)); 
OpGrad.Commit();

As an additional test, we create the gradient-matrix and verify that 
its transpose 
is equal to the negative **MtxDiv**-matrix:

In [ ]:
var MtxGrad = OpGrad.ComputeMatrix(TestMapping, null, TrialMapping); 
var Test2   = MtxGrad.Transpose() - MtxDiv_strong; 
Test2.InfNorm()

8.881784197001252E-16

### The complete Poisson-system

**Assembly of the system**

We also need the identity-matrix in the top-left corner 
of the Poisson-system:

In [ ]:
public class Identity :  
        BoSSS.Foundation.IVolumeForm  
{ 
    public IList<string> ParameterOrdering {  
        get { return new string[0]; }  
    } 
 
    public string component;  
 
    public IList<String> ArgumentOrdering {  
        get { return new string[] { component }; }  
    } 
 
    public TermActivationFlags VolTerms { 
        get { 
            return TermActivationFlags.AllOn; 
        } 
    } 
 
    public double VolumeForm(ref CommonParamsVol cpv,  
           double[] U, double[,] GradU,  
           double V, double[] GradV) { 
        return U[0]*V;            
    } 
}


We are going to implement the linear Poisson-operator
$$
    \left[ \begin{array}{ccc}
     1           &  0         & \partial_x \\
     0           &  1         & \partial_y \\
     -\partial_x & -\partial_y & 0 
\end{array} \right]
\cdot 
\left[ \begin{array}{c} \sigma_0 \\ \sigma_1 \\ u \end{array} \right]
= 
\left[ \begin{array}{c} c_0 \\ c_1 \\ c_2 \end{array} \right]
$$
The variables $c_0$, $c_1$ and $c_2$, which correspond to the 
test functions are also called co-domain variables of the operator.
We are using the negative divergence, since this will lead to a 
symmetric matrix, instead of a anti-symmetric one.
By doing so, we can e.g. use a Cholesky-factorization to determine 
whether the system is definite or not.

In [ ]:
var OpPoisson = new SpatialOperator(3, 3,  
                      QuadOrderFunc.Linear(), 
                      "sigma1", "sigma2", "u", // the domain-variables 
                      "c1", "c2", "c3"); //       the co-domain variables  
/// Now we add all required components to \code{OpPoisson}:

// == TODO == add the EquationComponents for the complete Poisson-system
OpPoisson.EquationComponents["c1"].Add(new Gradient_d(0)); 
OpPoisson.EquationComponents["c1"].Add(new Identity() { component = "sigma1" }); 
OpPoisson.EquationComponents["c2"].Add(new Gradient_d(1)); 
OpPoisson.EquationComponents["c2"].Add(new Identity() { component = "sigma2" }); 
OpPoisson.EquationComponents["c3"].Add(new Divergence_strong()); 
OpPoisson.Commit();

We create mappings $[\sigma_1, \sigma_2, u ]$:
three different combinations of DG orders will be investigated:

- equal order: the same polynomial degree for $u$ and $\vec{\sigma}$
- mixed order: the degree of $u$ is lower than the degree 
      of $\vec{\sigma}$.
- `strange' order: the degree of $u$ is higher than the degree of 
      $\vec{\sigma}$.


In [ ]:
var b3         = new Basis(grd2D, 3); 
var b2         = new Basis(grd2D, 2); 
var b4         = new Basis(grd2D, 4); 
var EqualOrder = new UnsetteledCoordinateMapping(b3,b3,b3); 
var MixedOrder = new UnsetteledCoordinateMapping(b4,b4,b3); 
var StrngOrder = new UnsetteledCoordinateMapping(b2,b2,b3);

In [ ]:
var MtxPoisson_Equal = OpPoisson.ComputeMatrix(EqualOrder, null, EqualOrder); 
var MtxPoisson_Mixed = OpPoisson.ComputeMatrix(MixedOrder, null, MixedOrder); 
var MtxPoisson_Strng = OpPoisson.ComputeMatrix(StrngOrder, null, StrngOrder);

We show that the matrices are symmetric 
(use e.g. **SymmetryDeviation(...)**), but indefinite
(use e.g. **IsDefinite(...)**).

In [ ]:
double symDev_Equal = MtxPoisson_Equal.SymmetryDeviation();
symDev_Equal

2.0920765120280294E-14

In [ ]:
double symDev_Mixed = MtxPoisson_Mixed.SymmetryDeviation();
symDev_Mixed

3.755329380794592E-14

In [ ]:
double symDev_Strng = MtxPoisson_Strng.SymmetryDeviation();
symDev_Strng

2.090688733247248E-14

In [ ]:
MtxPoisson_Equal.IsDefinite()

False

In [ ]:
MtxPoisson_Mixed.IsDefinite()

False

In [ ]:
MtxPoisson_Strng.IsDefinite()

False

In [ ]:
NUnit.Framework.Assert.LessOrEqual(symDev_Equal, 1.0e-8);
NUnit.Framework.Assert.LessOrEqual(symDev_Mixed, 1.0e-8);
NUnit.Framework.Assert.LessOrEqual(symDev_Strng, 1.0e-8);

### Algebraic reduction

Since the top-left corner of our matrix 
$$
\left[ \begin{array}{cc}
1   & B \\
B^T & 0 
\end{array} \right]
$$
is actually very easy to eliminate the variable $\vec{\sigma}$ from our system algebraically. 
The matrix of the reduces system is obviously $B^T \cdot B$.

From the mapping, we can actually obtain index-lists for each variable,
which can then be used to extract sub-matrices from 
**MtxPoisson\_Equal**, **MtxPoisson\_Mixed**, resp. 
**MtxPoisson\_Strng**.

In [ ]:
long[] SigmaIdx_Equal = EqualOrder.GetSubvectorIndices(true, 0,1); 
long[] uIdx_Equal     = EqualOrder.GetSubvectorIndices(true, 2); 
long[] SigmaIdx_Mixed = MixedOrder.GetSubvectorIndices(true, 0,1); 
long[] uIdx_Mixed     = MixedOrder.GetSubvectorIndices(true, 2); 
long[] SigmaIdx_Strng = StrngOrder.GetSubvectorIndices(true, 0,1); 
long[] uIdx_Strng     = StrngOrder.GetSubvectorIndices(true, 2);

The extraction of the sub-matrix and the elimination, for the equal order: 

In [ ]:
var MtxPoissonRed_Equal =  
      MtxPoisson_Equal.GetSubMatrix(uIdx_Equal, SigmaIdx_Equal)  // -Divergence 
    * MtxPoisson_Equal.GetSubMatrix(SigmaIdx_Equal, uIdx_Equal); // Gradient

Finally, we also
create the reduced system for the mixed and the strange 
order, test for the definiteness of the reduced system.

Equal and mixed order are positive definite, while the strange order
is indefinite - a clear indication that something ist wrong:

In [ ]:
var MtxPoissonRed_Mixed =  
      MtxPoisson_Mixed.GetSubMatrix(uIdx_Mixed, SigmaIdx_Mixed)  // -Divergence 
    * MtxPoisson_Mixed.GetSubMatrix(SigmaIdx_Mixed, uIdx_Mixed); // Gradient   
var MtxPoissonRed_Strng =  
      MtxPoisson_Strng.GetSubMatrix(uIdx_Strng, SigmaIdx_Strng)  // -Divergence 
    * MtxPoisson_Strng.GetSubMatrix(SigmaIdx_Strng, uIdx_Strng); // Gradient

In [ ]:
bool isdef_red_Equal = MtxPoissonRed_Equal.IsDefinite();
isdef_red_Equal

True

In [ ]:
bool isdef_red_Mixed = MtxPoissonRed_Mixed.IsDefinite();
isdef_red_Mixed

True

In [ ]:
bool isdef_red_Strng = MtxPoissonRed_Strng.IsDefinite();
isdef_red_Strng

False

In [ ]:
NUnit.Framework.Assert.IsTrue(isdef_red_Equal);
NUnit.Framework.Assert.IsTrue(isdef_red_Mixed);
NUnit.Framework.Assert.IsFalse(isdef_red_Strng);

We compute the condition number of all three matrices; we observe that
the mixed as well as the equal-order discretization result give rather 
moderate condition numbers. 

For the strange orders, the condition number
of the system is far to high:

In [ ]:
double condest_Mixed = MtxPoissonRed_Mixed.condest();
condest_Mixed

7016.828053419739

In [ ]:
double condest_Equal = MtxPoissonRed_Equal.condest();
condest_Equal

3577.101349907239

In [ ]:
double condest_Strng = MtxPoissonRed_Strng.condest();
condest_Strng

7.405467484617866E+18

In [ ]:
NUnit.Framework.Assert.LessOrEqual(condest_Mixed, 1e5);
NUnit.Framework.Assert.LessOrEqual(condest_Equal, 1e5);
NUnit.Framework.Assert.Greater(condest_Strng, 1e10);